# 选题二：语言模型先验（Colab + 免费 T4）

**这个 notebook 只做一件事**：在 Colab 的 T4 上，用 vLLM 批量生成语言模型先验，
并把缓存与结果通过 git 回传仓库。其余分析（评测、融合实验、画图、出文档）都在本机做。

**为什么用 git 而不是上传压缩包**：仓库每次改动只需要 `git pull` 一行，
不会出现"压缩包版本和本地不一致""路径分隔符在 Linux 下失效"这类问题；
结果与缓存也随提交回到仓库，导师和同组同学能直接看到。

**一次性准备**

1. 把仓库推到 GitHub 或 Gitee（本机执行一次）：
   `git remote add origin <仓库地址>` + `git push -u origin <分支>`；
2. 在 Colab 左侧密钥面板里配两个 Secret（没有面板就跳过 HF 那个，不影响 Qwen）：
   * `HF_TOKEN`：Hugging Face read token，避免下载限速；
   * `GIT_TOKEN`：仓库的写权限 token（GitHub 用 Personal Access Token，
     Gitee 用私人令牌），用于把结果推回去；
3. 如果仓库是私有的，克隆时把 `GIT_TOKEN` 拼进 URL（下一步的代码已经这么写）。

**每个会话的流程**：配置参数 → 拉仓库 → 一键搭环境 → 探测模型 → 生成 → 回传 → 校验。
整个环境（含 vLLM）在 Colab 上是临时的，所以每个会话都要重跑第 5 格；
但**生成结果是断点续跑的**，缓存已在仓库里，重复的部分会自动跳过。

## 1. 参数配置（只改这一格）

In [ ]:
# 仓库地址（已按你的仓库填好，一般不用改）
REPO_URL = 'https://github.com/yyyyyyy171/causal-llm.git'
BRANCH   = 'yyyyyyy171/topic2'

# 本会话要跑的网络，按变量数（变量对数量）从小到大排：
# 先跑完小的，中途掉线也已经拿到完整结果，不会白跑。
NETWORKS = ['ASIA', 'CHILD', 'INSURANCE', 'ALARM', 'HAILFINDER']

# 冒烟测试用哪个网络（只跑 4 对 × 2 次，确认通路后再开批量）
SMOKE_NETWORK = 'ASIA'

# 生成参数：冒烟与批量共用同一份。
# 缓存按“生效后的采样参数指纹”分档，两处参数不一致会让冒烟的结果作废 ——
# 这是故意的防呆，不是 bug：参数变了就该重算。
COMMON_ARGS = [
    '--task', 'direction',
    '--runs', '5',
    '--enforce-eager',
    '--gpu-memory-utilization', '0.85',
    '--max-model-len', '2048',
    '--max-new-tokens', '384',
]

# 仓库与虚拟环境位置（Colab 固定）
REPO = '/content/Causal-LLM'
VENV = '/content/venv'

## 2. 确认显卡与驱动

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!python -c "import sys; print('python', sys.version.split()[0])"

## 3. 拉取或更新仓库

第一次是 `clone`，之后每个会话也是 `clone`（虚拟机是新的）；
要是想在同一会话里同步最新改动，把这格改成 `git pull` 那句即可。

In [ ]:
import os, subprocess

# 关键：先离开可能被删除的仓库目录。
# 之前有格执行过 os.chdir('/content/Causal-LLM')，而删除仓库时进程的
# 当前工作目录就落在里面，删掉之后 git 会直接报
# "Unable to read current working directory: No such file or directory"。
os.chdir('/content')

# 防呆：占位符没替换的话，GitHub/Gitee 只会回一个 400，看不出真正原因
assert '<' not in REPO_URL and REPO_URL.endswith('.git'), (
    '请先把 REPO_URL 换成你自己的仓库地址，例如 '
    'https://github.com/yourname/causal-llm.git ；当前是 ' + REPO_URL
)

try:
    from google.colab import userdata
    token = userdata.get('GIT_TOKEN') or ''
except Exception:
    token = ''

# 从网页复制 token 常会带上换行/空格，不清掉的话 URL 就是坏的
# （之前就是这样：https://ghp_xxx\\r\\n@github.com/... 直接 clone 失败）
token = ''.join(token.split())

# 凭据只写进虚拟机的 ~/.git-credentials，由 git 自己读取：
#   * 不写进 remote URL —— 失败时 git 不会把 token 打进错误信息；
#   * 不放进命令行参数 —— 不会出现在进程列表和日志里；
#   * 该文件在 Colab 临时虚拟机里，不会被提交进仓库。
if token:
    with open(os.path.expanduser('~/.git-credentials'), 'w', encoding='utf-8') as handle:
        handle.write(f'https://{token}@github.com\n')
    subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
    print(f'已配置 git 凭据（token 长度 {len(token)}，不打印内容）')
else:
    print('没有配置 GIT_TOKEN：可以克隆公开仓库，但无法把结果推回仓库')

def on_expected_branch(repo):
    # 先确认停在期望分支上：detached HEAD 会让 git pull 直接失败
    result = subprocess.run(
        ['git', '-C', repo, 'rev-parse', '--abbrev-ref', 'HEAD'],
        capture_output=True, text=True)
    return result.stdout.strip() == BRANCH


if os.path.exists(REPO) and not on_expected_branch(REPO):
    # 之前遇到过：仓库处于 detached HEAD，git pull --rebase 报
    # "You are not currently on a branch"，于是代码永远停在旧版本。
    # 虚拟机里的仓库是可丢弃的（数据都在远端），直接重新克隆最干净。
    import shutil
    print('仓库不在', BRANCH, '分支上（可能是 detached HEAD），重新克隆 ...')
    shutil.rmtree(REPO)

if os.path.exists(REPO):
    # Colab 里通常会先产生本地提交（生成的缓存/结果），历史与远端分叉，
    # --ff-only 会直接失败且什么都不做。这里用 rebase 合并，并且**把结果打印出来**：
    # 之前用 check=False 静默吞掉了失败，导致"以为拉到最新、其实脚本还是旧的"。
    print('仓库已存在，拉取最新改动 ...')
    result = subprocess.run(
        ['git', '-C', REPO, 'pull', '--rebase', '--autostash'],
        capture_output=True, text=True)
    print(result.stdout.strip() or result.stderr.strip())
    if result.returncode != 0:
        print('!! 拉取失败：上面的输出说明了原因，处理完再继续，否则跑的是旧代码')
else:
    # 克隆用干净的地址；公开仓库不需要 token，私有仓库由上面的凭据自动补上
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, REPO], check=True)

# 提交身份（Colab 虚拟机里需要现配）
subprocess.run(['git', '-C', REPO, 'config', 'user.email', 'colab@local'])
subprocess.run(['git', '-C', REPO, 'config', 'user.name', 'colab'])

# 自证代码是最新的：脚本里应该含这几处修复，缺任何一个后面都是白跑。
# 之前吃过亏——pull 静默失败，脚本还是旧版，跑了两小时才发现。
script_path = os.path.join(REPO, 'scripts', 'run_llm_prior_vllm.py')
script_text = open(script_path, encoding='utf-8').read()
checks = {
    'chat_template_kwargs': 'chat_template_kwargs' in script_text,
    'finish_reason': 'finish_reason' in script_text,
    '_read_field': '_read_field' in script_text,
    'params_fingerprint': 'params_fingerprint' in script_text,
}
print('脚本版本自检:', checks)
assert all(checks.values()), (
    '脚本不是最新版，缺 ' + str([k for k, v in checks.items() if not v])
    + '；先解决上一格打印出来的拉取问题，否则生成结果不可用'
)

print(subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-3'],
                     capture_output=True, text=True).stdout)
print('已有缓存:')
for network in sorted(os.listdir(os.path.join(REPO, 'data'))):
    cache_dir = os.path.join(REPO, 'data', network, 'llm_cache')
    if os.path.isdir(cache_dir):
        for name in sorted(os.listdir(cache_dir)):
            with open(os.path.join(cache_dir, name), encoding='utf-8') as handle:
                print(f'  {network}/{name}: {sum(1 for _ in handle)} 条')

## 4. 一键搭建环境

环境由 `scripts/setup_colab.sh` 负责：建隔离虚拟环境、固定 vLLM 版本，
并在驱动低于 580 时自动把 torch 换成 CUDA 12.1 版本
（CUDA 13 的 torch 需要驱动 ≥580，Colab 的 T4 常常达不到，
这正是 vLLM 报 `Engine core initialization failed` 的常见原因）。

In [ ]:
!cd /content/Causal-LLM && bash scripts/setup_colab.sh

## 5. 登录 Hugging Face（可选）

`Qwen/Qwen3-8B-AWQ` 是公开仓库，匿名就能下载；配上 `HF_TOKEN` 只是避免限速。

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
    from huggingface_hub import login, whoami
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('已登录:', whoami().get('name'))
except Exception as error:
    print('未配置 HF_TOKEN，匿名下载（Qwen 系列可以忽略）:', type(error).__name__)

## 6. 选模型

优先 8B 的 4bit 量化版——文档要求 8B，而 T4 的 16GB 放不下纯 fp16 的 8B
（约 16GB 权重），放 AWQ 版（约 5.7GB）很从容。

In [ ]:
from huggingface_hub import model_info

CANDIDATES = [
    'Qwen/Qwen3-8B-AWQ',           # 8B 4bit，已确认存在（下载量 200 万+）
    'Qwen/Qwen3-4B-Instruct-2507', # 4B fp16，兜底（约 8GB）
]

MODEL = None
for candidate in CANDIDATES:
    try:
        model_info(candidate)
        MODEL = candidate
        print('可用:', candidate)
        break
    except Exception as error:
        print('不可用:', candidate, f'({type(error).__name__})')

print()
print('本会话将使用:', MODEL)

## 7. 冒烟测试（先跑 8 条，确认通路）

只跑 4 对变量 × 2 次，几十秒就能看出"生成长度有没有真的生效、解析率是不是 0"。
参数与批量那格完全一致，所以这 8 条记录会被批量阶段直接复用，不算白跑。

In [ ]:
import subprocess

assert MODEL, '上一格没探测到可用模型'

command = [
    f'{VENV}/bin/python', 'scripts/run_llm_prior_vllm.py',
    '--network', SMOKE_NETWORK,
    '--mode', 'real',
    '--limit', '4',            # 只跑前 4 对（4 对 × 5 次 = 20 条，约 10 秒）
    '--model', MODEL,
    *COMMON_ARGS,
]

print('运行:', ' '.join(command))
print()
subprocess.run(command, check=True, cwd=REPO)

## 8. 批量跑完全部网络（断点续跑 + 每个网络回传一次）

这一格是整条流程的主体，可以放着跑几个小时。三个设计点：

* **断点续跑**：缓存按 `(网络, 模式, 变量对, 第几次采样)` 逐条落盘，
  跑过的不会重跑。掉线后重开虚拟机、重跑这一格，会从断点接上；
* **逐网络回传**：每跑完一个网络就 `commit + push`。按变量对从小到大执行，
  所以哪怕只跑完 ASIA 和 CHILD 就掉线，这两个网络的结果也已经安全躺在仓库里；
* **立刻自检**：每个网络跑完都打印解析率与 `finish_reason` 分布。
  解析率 0 或 finish_reason 全是 `length` 说明生成被截断，当场就能发现，
  不用等三个小时跑完才去看。

`git pull` 放在开头：本机推过来的修复会被这一格带上，不需要手动操作。

In [ ]:
import json, os, re, subprocess, time, urllib.request

# 兼容层：这一格不依赖前面的配置格也能跑。
# 前面的配置格（第 1 格）定义过就用它，没定义就用这里的默认值 ——
# 这样把这一格单独粘进一个旧会话（变量名还是老的 NETWORK）也不会 NameError。
NETWORKS = globals().get('NETWORKS') or [
    'ASIA', 'CHILD', 'INSURANCE', 'ALARM', 'HAILFINDER']
COMMON_ARGS = globals().get('COMMON_ARGS') or [
    '--task', 'direction',
    '--runs', '5',
    '--enforce-eager',
    '--gpu-memory-utilization', '0.85',
    '--max-model-len', '2048',
    '--max-new-tokens', '384',
]
REPO_URL = globals().get('REPO_URL') or 'https://github.com/yyyyyyy171/causal-llm.git'
REPO     = globals().get('REPO')     or '/content/Causal-LLM'
BRANCH   = globals().get('BRANCH')   or 'yyyyyyy171/topic2'
VENV     = globals().get('VENV')     or '/content/venv'
MODEL    = globals().get('MODEL')    or 'Qwen/Qwen3-8B-AWQ'

assert os.path.isdir(REPO), REPO + ' 不存在，先跑“拉取仓库”那一格'
assert os.path.isdir(VENV), VENV + ' 不存在，先跑“一键搭建环境”那一格'

# 与 run_llm_prior.py 的 model_suffix 保持一致：文件名里的模型后缀
MODEL_TAG = re.sub(r'[^a-z0-9]+', '-', MODEL.split('/')[-1].lower()).strip('-')
SUFFIX = '' if MODEL == 'Qwen/Qwen3-8B' else '_' + MODEL_TAG
print('模型:', MODEL, '| 结果文件后缀:', SUFFIX)

# ---------------------------------------------------------------- 凭据与自检
# 每次都从密钥面板重新读 token，不用 kernel 里的旧变量：
# 你在面板里换了新 token，旧值还留在内存里，结果就是“明明改了还是推不上去”。
try:
    from google.colab import userdata
    token = ''.join((userdata.get('GIT_TOKEN') or '').split())
except Exception as error:
    token = ''
    print('读不到 Colab 的 GIT_TOKEN Secret:', type(error).__name__)

if not token:
    raise RuntimeError(
        '没有 GIT_TOKEN，拒绝开工：这几小时的结果只会留在临时虚拟机里，'
        '会话一结束就没了。\n'
        '  在 Colab 左侧“钥匙”图标 -> Secrets 里添加 GIT_TOKEN，'
        '值填 GitHub 的 PAT，然后重跑这一格。')

owner, name = REPO_URL.rstrip('/').replace('.git', '').split('/')[-2:]

# 凭据必须写成 用户名:token 两段。
# 只写 token@github.com 的话，git 会解析成“用户名 = token、密码 = 空”，
# GitHub 回 401，git 接着去问用户名，最后报 could not read Username ——
# 看起来像“没配 token”，其实是格式错了。
with open(os.path.expanduser('~/.git-credentials'), 'w', encoding='utf-8') as handle:
    handle.write('https://%s:%s@github.com\n' % (owner, token))
subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
print('已写入 git 凭据（用户名 %s，token 长度 %d，不打印内容）' % (owner, len(token)))

# 开跑前先把“能不能推回去”验掉。几分钟就能发现的问题，不该等三小时。
request = urllib.request.Request(
    'https://api.github.com/repos/%s/%s' % (owner, name),
    headers={'Authorization': 'Bearer ' + token,
             'Accept': 'application/vnd.github+json'})
try:
    with urllib.request.urlopen(request, timeout=20) as response:
        repo_info = json.load(response)
    has_push = (repo_info.get('permissions') or {}).get('push')
except Exception as error:
    detail = error.read()[:200].decode('utf-8', 'replace') if hasattr(error, 'read') else ''
    # 注意：换行拼接时不要把 % 放在行首 —— 逐行检查 notebook 的脚本会把
    # 行首的 % 当成 IPython 行魔法（%pip 之类）而误报语法错误。
    message = 'GitHub 拒绝了 token（' + type(error).__name__ + '）。' + detail
    message += '\n  到 https://github.com/settings/tokens 新建一个 classic token 并勾选 repo，'
    message += '\n  把 Colab 的 GIT_TOKEN 换成新值，再重跑这一格。'
    raise RuntimeError(message)

print('推送权限自检: push =', has_push)
if not has_push:
    raise RuntimeError(
        'token 能读这个仓库，但没有写权限。\n'
        '  用 fine-grained token 的话，把 Contents 设成 Read and write；\n'
        '  或者干脆用 classic token 勾 repo。')


def git_sync_push(message):
    # 提交并推送。远端有新提交时先 rebase 再推一次，避免"本地能跑、推不上去"。
    # token 全程不进命令行、不进 URL —— 泄露过一次，教训在这里。
    subprocess.run(['git', '-C', REPO, 'add', '-A'], check=True)
    commit = subprocess.run(['git', '-C', REPO, 'commit', '-m', message],
                            capture_output=True, text=True)
    print((commit.stdout or commit.stderr).strip()[-300:])
    for attempt in (1, 2):
        push = subprocess.run(['git', '-C', REPO, 'push', 'origin', BRANCH],
                              capture_output=True, text=True)
        if push.returncode == 0:
            print(f'已推送到 {BRANCH}（第 {attempt} 次尝试）')
            return True
        # 打印完整报错：这里吞掉细节的话，下一轮就得靠猜
        print('推送失败:', (push.stdout + push.stderr).strip()[-800:])
        rebase = subprocess.run(
            ['git', '-C', REPO, 'pull', '--rebase', '--autostash', 'origin', BRANCH],
            capture_output=True, text=True)
        print('rebase:', (rebase.stdout + rebase.stderr).strip()[-800:])
    # 推不上去就停下：继续跑只会让更多结果困在临时虚拟机里
    raise RuntimeError(
        f'推送失败两次，已停下。结果都还在 {REPO} 里，'
        '修好凭据后重跑这一格（跑过的不会重算）。')


def report(network, mode):
    # 跑完立刻体检：解析率 + 结束原因。这里不通过，后面所有分析都是废的。
    path = os.path.join(
        REPO, 'data', network, f'{network}_LLM_prior_{mode}{SUFFIX}.json')
    if not os.path.exists(path):
        print(f'  {network}/{mode}: 没有产出文件，说明这一步没跑到')
        return
    with open(path, encoding='utf-8') as handle:
        records = json.load(handle)
    runs = [run for item in records for run in item['runs']]
    ok = sum(1 for run in runs if run.get('parse_status') == 'success')
    reasons = {}
    for run in runs:
        reasons[run.get('finish_reason')] = reasons.get(run.get('finish_reason'), 0) + 1
    rate = ok / len(runs) if runs else 0.0
    print(f'  {network}/{mode}: 解析 {ok}/{len(runs)} = {rate:.1%}，结束原因 {reasons}')
    if reasons.get('length', 0) > 0.2 * max(len(runs), 1):
        print('    !! 大量 length 截断：生成长度又被丢掉了，先别继续跑大网络')


# 先同步本机推过来的最新代码（含脚本修复与新网络配置）
subprocess.run(['git', '-C', REPO, 'pull', '--rebase', '--autostash'],
               capture_output=True, text=True)

PAIRS = {'ASIA': 28, 'CHILD': 190, 'INSURANCE': 351, 'ALARM': 666, 'HAILFINDER': 1540}
print()
print('预计工作量（每个网络 real + anon 各 5 次采样，按实测 2.2 条/秒估）：')
for name in NETWORKS:
    count = PAIRS.get(name, 0) * 5 * 2
    print(f'  {name:<11s} {count:5d} 条，约 {count / 2.2 / 60:3.0f} 分钟')
print()

batch_started = time.time()

for network in NETWORKS:

    started = time.time()
    command = [
        f'{VENV}/bin/python', 'scripts/run_llm_prior_vllm.py',
        '--network', network,
        '--mode', 'both',        # 真实名 + 匿名，匿名用于语义消融
        '--model', MODEL,
        *COMMON_ARGS,
    ]

    print('=' * 72)
    print(f'开始 {network}（real + anon）', flush=True)
    result = subprocess.run(command, cwd=REPO)
    elapsed = (time.time() - started) / 60
    print(f'{network} 退出码 {result.returncode}，耗时 {elapsed:.1f} 分钟')

    for mode in ('real', 'anon'):
        report(network, mode)

    git_sync_push(f'Colab T4 生成 {network} 先验（{MODEL}，real+anon）')

print('=' * 72)
print(f'本次会话批量阶段共耗时 {(time.time() - batch_started) / 60:.1f} 分钟')

## 9. 校验结构化输出

In [ ]:
!cd /content/Causal-LLM && python scripts/validate_json_output.py